# GTEx simulations

Exclude regions with too few variants.

## 1. Subset GTEx to participants of European ancestry

In [ ]:
#!/bin/bash
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J gtex_ancestry
#BSUB -n 8
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=8000]
#BSUB -W 24:00
#BSUB -o gtex_ancestry.%J.out
#BSUB -e gtex_ancestry.%J.err
set -euo pipefail
module load plink2
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH

GTEX=/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/genotype
KG=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_all
OUT=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_ancestry
SCRIPT_DIR=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_ancestry
mkdir -p $OUT && cd $OUT
P="plink2 --threads 8 --memory 56000 --allow-extra-chr --output-chr 26"
IDS='@:#:$1:$2'   # chr:pos:alleles in sorted order -> matches regardless of ref/alt order

# ---- 0. 1000G: decompress pgen (pvar.zst is read directly with 'vzs') ----
[ -f kg.pgen ] || plink2 --zst-decompress $KG/all_hg38.pgen.zst > kg.pgen
ln -sf $KG/all_hg38.pvar.zst kg.pvar.zst
ln -sf $KG/all_hg38.psam     kg.psam

# ---- 1. 1000G QC: autosomal biallelic ACGT SNPs, common, uniform IDs ----
$P --pfile kg vzs --chr 1-22 --snps-only just-acgt --max-alleles 2 \
   --set-all-var-ids "$IDS" --rm-dup exclude-all \
   --maf 0.05 --geno 0.02 --make-pgen --out kg_qc

# ---- 2. GTEx QC per chromosome, same ID scheme ----
: > gtex_ids.txt
for c in $(seq 1 22); do
  $P --bfile $GTEX/GTEx_v8_genotype.$c --snps-only just-acgt --max-alleles 2 \
     --set-all-var-ids "$IDS" --rm-dup exclude-all --geno 0.05 \
     --make-pgen --out gtex_c$c
  grep -v '^#' gtex_c$c.pvar | cut -f3 >> gtex_ids.txt
done

# ---- 3. Shared variants, drop strand-ambiguous A/T and C/G ----
awk 'NR==FNR{g[$1]; next}
     !/^#/ && ($3 in g) { a=$4$5; if (a!="AT" && a!="TA" && a!="CG" && a!="GC") print $3 }' \
    gtex_ids.txt kg_qc.pvar > common_ids.txt

# long-range LD regions (approximate hg38): MHC, 8p23.1 inversion, 17q21.31 inversion
printf "6\t25000000\t34000000\tMHC\n8\t8000000\t12000000\tinv8p23\n17\t44500000\t47000000\tinv17q21\n" > longrange_hg38.txt

# ---- 4. LD pruning + unrelated 1000G samples ----
$P --pfile kg_qc --extract common_ids.txt --exclude range longrange_hg38.txt \
   --indep-pairwise 200 50 0.1 --out kg_prune
$P --pfile kg_qc --extract kg_prune.prune.in --king-cutoff 0.0884 --out kg_king

# ---- 5. PCA on unrelated 1000G with allele weights ----
$P --pfile kg_qc --extract kg_prune.prune.in --keep kg_king.king.cutoff.in.id \
   --nonfounders --maf 0.05 --freq counts --pca 10 allele-wts vcols=chrom,ref,alt --out kg_pca

# ---- 6. Project 1000G and GTEx with the SAME scoring (sums, so chromosomes add up) ----
SCORE="--read-freq kg_pca.acount --score kg_pca.eigenvec.allele 2 5 header-read no-mean-imputation variance-standardize cols=+scoresums --score-col-nums 6-15"
$P --pfile kg_qc --keep kg_king.king.cutoff.in.id $SCORE --out proj_kg
for c in $(seq 1 22); do
  $P --pfile gtex_c$c $SCORE --out proj_gtex_c$c
done

# ---- 7. Assign ancestry ----
Rscript "$SCRIPT_DIR/gtex_ancestry_assign.R"

In [ ]:
suppressMessages(library(data.table))
setwd("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_ancestry")
KG_PSAM  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_all/all_hg38.psam"
GTEX_FAM <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/gtex/GTEX_data/genotype/GTEx_v8_genotype.1.fam"
npc  <- 6      # PCs used for classification
k    <- 20     # nearest reference neighbours
pmin <- 0.9    # min neighbour fraction to call an ancestry

read_sc <- function(f) {
  d <- fread(f); setnames(d, sub("^#", "", names(d)))
  d[, c("IID", grep("^PC[0-9]+_SUM$", names(d), value = TRUE)), with = FALSE]
}

## projected scores
r <- read_sc("proj_kg.sscore")
g <- rbindlist(lapply(Sys.glob("proj_gtex_c*.sscore"), read_sc))
g <- g[, lapply(.SD, sum), by = IID]               # sum over chromosomes

psam <- fread(KG_PSAM); setnames(psam, sub("^#", "", names(psam)))
pop_col <- grep("^super", names(psam), ignore.case = TRUE, value = TRUE)[1]
r <- merge(r, psam[, .(IID, SuperPop = get(pop_col))], by = "IID")

## scale on reference, kNN ancestry fractions
pcs <- paste0("PC", 1:npc, "_SUM")
Rm  <- scale(as.matrix(r[, ..pcs]))
Gm  <- scale(as.matrix(g[, ..pcs]),
             center = attr(Rm, "scaled:center"), scale = attr(Rm, "scaled:scale"))
pops <- sort(unique(r$SuperPop))
frac <- t(apply(Gm, 1, function(x) {
  d <- sqrt(colSums((t(Rm) - x)^2))
  table(factor(r$SuperPop[order(d)[1:k]], levels = pops)) / k
}))
colnames(frac) <- paste0("frac_", pops)
g <- cbind(g, as.data.table(frac))
g[, ancestry := ifelse(apply(frac, 1, max) >= pmin,
                       pops[max.col(frac, ties.method = "first")], "ADMIXED/UNCERTAIN")]

## extra guard: within 6 SD of the 1000G EUR cloud on every PC used
E  <- Rm[r$SuperPop == "EUR", , drop = FALSE]
z  <- sweep(sweep(Gm, 2, colMeans(E)), 2, apply(E, 2, sd), "/")
g[, eur_within6sd := apply(abs(z) <= 6, 1, all)]
g[, EUR_final := ancestry == "EUR" & eur_within6sd]

## optional: combine with self-reported race (dbGaP phs000424 subject phenotypes)
# ph <- fread("GTEx_Subject_Phenotypes.GRU.txt.gz"); g <- merge(g, ph[, .(IID = SUBJID, RACE)], by = "IID", all.x = TRUE)
# g[, EUR_final := EUR_final & RACE == 3]   # confirm coding in the data dictionary

## outputs
fam  <- fread(GTEX_FAM, header = FALSE, col.names = c("FID", "IID", "PAT", "MAT", "SEX", "PHENO"))
keep <- merge(fam[, .(FID, IID)], g[EUR_final == TRUE, .(IID)], by = "IID")[, .(FID, IID)]
fwrite(keep, "gtex_EUR_keep.txt", sep = "\t", col.names = FALSE)   # plink --keep format
fwrite(g,    "gtex_ancestry_assignment.tsv", sep = "\t")

cat("GTEx samples:", nrow(g), "\n"); print(table(g$ancestry))
cat("EUR kept (kNN >=", pmin, "& within 6 SD):", nrow(keep), "\n")

## plot
cols <- c(AFR = "#E69F00", AMR = "#56B4E9", EAS = "#009E73", EUR = "#0072B2", SAS = "#CC79A7")
pdf("gtex_ancestry_PC.pdf", width = 11, height = 5)
par(mfrow = c(1, 2))
for (pp in list(c(1, 2), c(3, 4))) {
  plot(Rm[, pp], pch = 16, cex = 0.5, col = adjustcolor(cols[r$SuperPop], 0.4),
       xlab = paste0("PC", pp[1]), ylab = paste0("PC", pp[2]))
  points(Gm[, pp], pch = 3, cex = 0.6, col = ifelse(g$EUR_final, "black", "red"))
  legend("topright", c(names(cols), "GTEx EUR", "GTEx non-EUR"),
         col = c(cols, "black", "red"), pch = c(rep(16, 5), 3, 3), cex = 0.7, bty = "n")
}
dev.off()

In [1]:
library(tidyverse)
ancestry_result = read_tsv("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_ancestry/gtex_ancestry_assignment.tsv")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Rows: 838 Columns: 19
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (2): IID, ancestry
dbl (15): PC1_SUM, PC2_SUM, PC3_SUM, PC4_SUM, PC5_SUM, PC6_SUM, PC7_SUM, PC8...
lgl  (2): eur_within6sd, EUR_final

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [9]:
ancestry_result %>% filter(EUR_final) %>% dim()

[1] 693  19

In [ ]:
#!/bin/bash
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J gtex_EUR_trim
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=16000]
#BSUB -W 24:00
#BSUB -o gtex_EUR_trim.%J.out
#BSUB -e gtex_EUR_trim.%J.err
set -uo pipefail
module load plink2

GENO=/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/genotype
RANGE=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/snp_info_random
KEEP=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_ancestry/gtex_EUR_keep.txt   # FID IID, from the ancestry job
OUT=/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_EUR_trimed_random
mkdir -p $OUT

printf "region\tchr\tn_variants\tn_samples\n" > $OUT/summary.tsv
for X in $(seq 1 500); do
  range=$RANGE/snp_${X}_range.txt
  [ -s "$range" ] || { echo "region $X: no range file" >&2; continue; }

  chr=$(awk 'NF {c=$1; sub(/^chr/, "", c); print c; exit}' "$range")

  if plink2 --bfile $GENO/GTEx_v8_genotype.${chr} \
            --keep $KEEP \
            --extract range "$range" \
            --make-bed \
            --out $OUT/gtex_${X} > /dev/null; then
    printf "%s\t%s\t%s\t%s\n" $X $chr $(wc -l < $OUT/gtex_${X}.bim) $(wc -l < $OUT/gtex_${X}.fam) >> $OUT/summary.tsv
  else
    echo "region $X: plink2 failed (see $OUT/gtex_${X}.log)" >&2
  fi
done

## Simulation code

In [ ]:
[multi_ref_simulation]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 5
parameter: walltime = "100h"
parameter: mem = "120G"
parameter: numThreads = 2
parameter: n_causal = 1
# specify the number of causal variants
parameter: h2g = 0.05
parameter: total_h2g = True
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_multi_ref.rds'
task: trunk_workers = 4, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    file = ${_input:r}
    i <- file %>%
        str_remove("\\.bim$") %>%
        str_remove(".*/gtex_")

    # -----------------------------------------------------------------------------
    # Reference panel genotypes
    # -----------------------------------------------------------------------------
    gtex_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/gtex_", i)
    gtex_data <- plink_process(gtex_path)

    UKB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKB_data <- plink_process(UKB_path)

    kg_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/EUR_KG_", i)
    kg_data <- plink_process(kg_path, imiss = 0.1, maf = 0)

    # -----------------------------------------------------------------------------
    # UKB sketch: U_UKB = X_UKB' W   (p_UKB x B, raw allele coding)
    # -----------------------------------------------------------------------------
    W <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/random_vector/result/random_matrix_10000.rds")
    U_UKB <- crossprod(UKB_data$bed, W)

    # -----------------------------------------------------------------------------
    # ADSP sketch (pgen): read X and afreq, recover/scale to build U_ADSP
    # -----------------------------------------------------------------------------
    read_pvar_text <- function(pvar_path) {
        df <- as.data.frame(vroom(pvar_path, comment = "##", show_col_types = FALSE))
        colnames(df)[1] <- "chrom"
        data.frame(
            chrom = as.character(df$chrom),
            id    = as.character(df$ID),
            pos   = as.integer(df$POS),
            A2    = as.character(df$REF),
            A1    = as.character(df$ALT),
            stringsAsFactors = FALSE
        )
    }

    prefix <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes//ADSP_sketch_trimed/result/ADSP_", i)
    pvar   <- paste0(prefix, ".pvar")
    pgen   <- paste0(prefix, ".pgen")
    psam   <- paste0(prefix, ".psam")
    paths  <- list(pvar = pvar, pgen = pgen, psam = psam)

    all_variant_info <- read_tsv(paths$pvar)
    variant_idx      <- seq_len(nrow(all_variant_info))
    variant_info     <- all_variant_info[variant_idx, , drop = FALSE]

    psam <- as.data.frame(read_tsv(paths$psam, show_col_types = FALSE))
    colnames(psam)[1:2] <- c("FID", "IID")

    pgen_obj <- pgenlibr::NewPgen(paths$pgen)
    X <- pgenlibr::ReadList(pgen_obj, variant_subset = variant_idx, meanimpute = FALSE)
    rownames(X) <- psam$IID
    colnames(X) <- variant_info$ID

    chrom_adsp <- all_variant_info$`#CHROM`[1]

    afreq_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/chr", chrom_adsp,
                         "/ADSP.R4.EUR.chr", chrom_adsp, ".afreq")
    afreq <- read_tsv(afreq_path)

    afreq_ordered <- afreq %>% slice(match(colnames(X), ID))
    umin <- afreq_ordered %>% pull(U_MIN)
    umax <- afreq_ordered %>% pull(U_MAX)

    colnames(variant_info) <- c("chrom", "pos", "original_id", "A2", "A1")

    recover_U <- function(U_scaled, Umin, Umax) {
        sweep(
            sweep(U_scaled, 2, 2, "/"),
            2, (Umax - Umin), "*"
        ) + matrix(Umin, nrow(U_scaled), ncol(U_scaled), byrow = TRUE)
    }

    U_ADSP_raw <- X
    U_ADSP_rec <- recover_U(U_ADSP_raw, umin, umax)
    U_ADSP     <- t(scale(U_ADSP_rec, center = TRUE, scale = TRUE))

    ADSP_data <- list(bim = variant_info, U = U_ADSP)

    ## Load finished

    # -----------------------------------------------------------------------------
    # Align all panels to GTEx (target)
    # -----------------------------------------------------------------------------
    target_tb <- gtex_data$bim %>%
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    uniform <- function(target, reference, reference_LD = NULL, reference_U = NULL, flip_cols = FALSE) {
        qc_data <- allele_qc(
            target_data  = target,
            ref_variants = reference,
            col_to_flip  = if (flip_cols) c("Beta", "z") else NULL
        )
    
        sumstat_flipped <- qc_data$target_data_qced %>% arrange(pos)
        kept_vars       <- sumstat_flipped$variant_id
    
        out <- list(sumstat = sumstat_flipped)
    
        if (!is.null(reference_LD)) {
            out$LD <- reference_LD[kept_vars, kept_vars]
        }
        if (!is.null(reference_U)) {
            out$U <- reference_U[kept_vars, , drop = FALSE]
        }
    
        out
    }

    filter_vars <- function(input, variants) {
        sumstat   <- input$sumstat %>% filter(variants_id_original %in% variants)
        kept_vars <- sumstat %>% pull(variant_id)

        out <- list(sumstat = sumstat)

        if (!is.null(input$LD)) {
            out$LD <- input$LD[kept_vars, kept_vars, drop = FALSE]
        }
        if (!is.null(input$U)) {
            out$U <- input$U[kept_vars, , drop = FALSE]
        }

        out
    }

    gtex_LD <- get_cormat(gtex_data$bed)
    kg_LD   <- get_cormat(kg_data$bed)
    
    insample_input <- list(
        sumstat = target_tb %>% arrange(pos),
        LD      = gtex_LD[target_tb$variant_id, target_tb$variant_id]
    )
    kg_input   <- uniform(target = target_tb, reference = kg_data$bim,   reference_LD = kg_LD)
    UKB_input  <- uniform(target = target_tb, reference = UKB_data$bim,  reference_U  = U_UKB)
    ADSP_input <- uniform(target = target_tb, reference = ADSP_data$bim, reference_U  = U_ADSP)
    
    shared_vars <- Reduce(intersect, list(
        insample_input$sumstat$variants_id_original,
        UKB_input$sumstat$variants_id_original,
        kg_input$sumstat$variants_id_original,
        ADSP_input$sumstat$variants_id_original
    ))
    
    insample_input <- filter_vars(insample_input, shared_vars)
    UKB_input      <- filter_vars(UKB_input,      shared_vars)
    kg_input       <- filter_vars(kg_input,       shared_vars)
    ADSP_input     <- filter_vars(ADSP_input,     shared_vars)
    
    # -----------------------------------------------------------------------------
    # Simulate phenotype + summary stats
    # -----------------------------------------------------------------------------
    Xmat <- gtex_data$bed[, shared_vars, drop = FALSE]
    if (ncol(Xmat) < 500) stop("Too few SNPs — terminating job")
    
    n_samples <- nrow(Xmat)
    p         <- ncol(Xmat)
    
    ntrait  <- 1
    ncausal <- ${n_causal}
    h2g     <- ${h2g}
    
    causal_index <- sample_causal_index(Xmat = Xmat, ncausal = ncausal)
    B_mat        <- matrix(0, nrow = p, ncol = ntrait)
    beta         <- sim_beta_fix_variant(G = Xmat, causal_index = causal_index, is_h2g_total = FALSE)
    B_mat[, 1]   <- beta
    
    phenotype <- sim_multi_traits(G = Xmat, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P
    
    target_tb <- calculate_sumstat(Xmat, as.numeric(phenotype[, 1])) %>%
      separate(SNP, into = c("chrom", "pos", "A2", "A1"), sep = ":", remove = FALSE) %>%
      mutate(
        chrom      = as.numeric(str_remove(chrom, "^chr")),
        pos        = as.integer(pos),
        original_z = z
      ) %>%
      mutate(
        variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
        variant_id           = variants_id_original
      )
    
    insample_input$sumstat <- target_tb
    
    # -----------------------------------------------------------------------------
    # Re-QC each external panel against simulated z-scores (flip_cols = TRUE)
    # -----------------------------------------------------------------------------
    kg_input <- uniform(
        target       = target_tb,
        reference    = kg_input$sumstat %>% dplyr::select(-variant_id),
        reference_LD = kg_input$LD,
        flip_cols    = TRUE
    )
    
    UKB_input <- uniform(
        target       = target_tb,
        reference    = UKB_input$sumstat %>% dplyr::select(-variant_id),
        reference_U  = UKB_input$U,
        flip_cols    = TRUE
    )
    
    ADSP_input <- uniform(
        target       = target_tb,
        reference    = ADSP_input$sumstat %>% dplyr::select(-variant_id),
        reference_U  = ADSP_input$U,
        flip_cols    = TRUE
    )
    output_list <- list(
        insample_input = insample_input,
        kg_input       = kg_input,
        ADSP_input     = ADSP_input,
        UKB_input      = UKB_input,
        true_vars      = causal_index,
        n_samples      = n_samples,
        n_snp          = p,
        input_path     = gtex_path
    )
    
    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output_list, data_out_path)

In [ ]:
work_dir="/home/sunh4/rss_qc/output/trim_genotype/ROSMAP_trimed"
mkdir -p ${work_dir}/{code,log,result,tmp}
cd ${work_dir}/code

cat << 'EOF' > base_script.sh
#!/bin/bash -l
#SBATCH -t 28:00:00
#SBATCH --mem=30G
#SBATCH -J ROSMAP_trim
#SBATCH -o /home/sunh4/rss_qc/output/trim_genotype/ROSMAP_trimed/log/trim.%j.out
#SBATCH -e /home/sunh4/rss_qc/output/trim_genotype/ROSMAP_trimed/log/trim.%j.err


export PATH=$HOME/miniforge3/envs/space/bin:$PATH

mapfile -t chroms < /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/chrom_region.txt

for i in {1..500}; do
    chrom="${chroms[$((i-1))]}"

    bfile_prefix="/data1/deyk/ftp_fgc_xqtl/ROSMAP/genotype/analysis_ready/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.${chrom}"
    rsid_file="/home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_${i}.txt"
    map_tsv="/home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_${i}.tsv"
    
    extract_file="/home/sunh4/rss_qc/output/trim_genotype/ROSMAP_trimed/tmp/snp_${i}.pvar_id.txt"
    out_prefix="/home/sunh4/rss_qc/output/trim_genotype/ROSMAP_trimed/result/ADSP_${i}"
    
    awk 'NR==FNR {keep[$1]=1; next}
         FNR>1 && ($2 in keep) {
             print "chr"$1":"$3"_"$5"_"$4
         }' "${rsid_file}" "${map_tsv}" | sort -u > "${extract_file}"
    
    if [ ! -s "${extract_file}" ]; then
        echo "No matched variants for i=${i}, chrom=${chrom}" >&2
        continue
    fi
    
    plink2 \
      --bfile "${bfile_prefix}" \
      --extract "${extract_file}" \
      --make-bed \
      --out "${out_prefix}"
done
EOF

chmod +x base_script.sh

In [ ]:
[multi_ref_GWAS]
parameter: genofile = paths
# One .bim file per region; each region is processed independently.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "80G"
parameter: numThreads = 2

# ---- Simulation knobs --------------------------------------------------------
parameter: n_causal = 1        # number of causal variants per region
parameter: h2g = 0.05          # per-region heritability
parameter: total_h2g = True    # interpret h2g as total (vs per-causal)

parameter: container = ""

input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_multi_ref.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    file = ${_input:r}
    i <- file %>%
        str_remove("\\.bim$") %>%
        str_remove(".*/gtex_")

    # -----------------------------------------------------------------------------
    # Reference panel genotypes
    # -----------------------------------------------------------------------------
    gtex_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/gtex_", i)
    gtex_data <- plink_process(gtex_path)

    UKB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKB_data <- plink_process(UKB_path)

    kg_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/EUR_KG_", i)
    kg_data <- plink_process(kg_path, imiss = 0.1, maf = 0)

    # -----------------------------------------------------------------------------
    # UKB sketch: U_UKB = X_UKB' W   (p_UKB x B, raw allele coding)
    # -----------------------------------------------------------------------------
    W <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/random_vector/result/random_matrix_10000.rds")
    U_UKB <- crossprod(UKB_data$bed, W)

    # ---- ADSP sketch panel from pgen + afreq --------------------------------
    # ADSP genotypes are stored as a scaled sketch X (B x p) in pgen format.
    # We need to (a) read X, (b) un-scale to original U coding using per-variant
    # (U_MIN, U_MAX) from the afreq file, then (c) column-standardize so the
    # sketch yields a correlation (not covariance) approximation.
    read_pvar_text <- function(pvar_path) {
        df <- as.data.frame(vroom(pvar_path, comment = "##", show_col_types = FALSE))
        colnames(df)[1] <- "chrom"
        data.frame(
            chrom = as.character(df$chrom),
            id    = as.character(df$ID),
            pos   = as.integer(df$POS),
            A2    = as.character(df$REF),
            A1    = as.character(df$ALT),
            stringsAsFactors = FALSE
        )
    }

    prefix <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes//ADSP_sketch_trimed/result/ADSP_", i)
    pvar   <- paste0(prefix, ".pvar")
    pgen   <- paste0(prefix, ".pgen")
    psam   <- paste0(prefix, ".psam")
    paths  <- list(pvar = pvar, pgen = pgen, psam = psam)

    # Read variant info from .pvar as text (pgenlibr::NewPvar has been flaky here).
    all_variant_info <- read_tsv(paths$pvar)
    variant_idx      <- seq_len(nrow(all_variant_info))
    variant_info     <- all_variant_info[variant_idx, , drop = FALSE]

    # Sample IDs.
    psam <- as.data.frame(read_tsv(paths$psam, show_col_types = FALSE))
    colnames(psam)[1:2] <- c("FID", "IID")

    # Genotype dosage matrix X (B x p): rows are sketch dimensions, cols are SNPs.
    pgen <- pgenlibr::NewPgen(paths$pgen)
    X <- pgenlibr::ReadList(pgen, variant_subset = variant_idx, meanimpute = FALSE)
    rownames(X) <- psam$IID
    colnames(X) <- variant_info$ID

    # Per-variant (U_MIN, U_MAX) for un-scaling the stored sketch.
    chrom_adsp      <- all_variant_info$`#CHROM`[1]
    afreq_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/chr", chrom_adsp,
                         "/ADSP.R4.EUR.chr", chrom_adsp, ".afreq")
    afreq <- read_tsv(afreq_path)

    afreq_ordered <- afreq %>% slice(match(colnames(X), ID))
    umin <- afreq_ordered %>% pull(U_MIN)
    umax <- afreq_ordered %>% pull(U_MAX)

    colnames(variant_info) <- c("chrom", "pos", "original_id", "A2", "A1")

    # Invert the storage transform: X_stored = 2 * (U - U_MIN) / (U_MAX - U_MIN).
    recover_U <- function(U_scaled, Umin, Umax) {
        sweep(
            sweep(U_scaled, 2, 2, "/"),
            2, (Umax - Umin), "*"
        ) + matrix(Umin, nrow(U_scaled), ncol(U_scaled), byrow = TRUE)
    }

    U_ADSP <- X
    U_rec  <- recover_U(U_ADSP, umin, umax)
    # Column-standardize, then transpose so U_ADSP is p x B (matches U_UKB).
    U_ADSP <- t(scale(U_rec, center = TRUE, scale = TRUE))

    ADSP_data <- list(bim = variant_info, U = U_ADSP)

    # ---- Done loading reference panels --------------------------------------

    # =========================================================================
    # 3. Align all panels to the UKB target (allele QC + variant intersection)
    # =========================================================================

    # Target variant table: UKB bim with a canonical "chr:pos:A2:A1" key.
    target_tb <- UKB_data$bim %>%
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    # uniform(): allele-QC the target against a reference panel, then carry along
    # whichever of (LD, U) the reference provides, subsetted to kept variants.
    # flip_cols toggles whether Beta/z columns are sign-flipped for strand fixes
    # (only meaningful once sumstats exist; pre-simulation we leave it FALSE).
    uniform <- function(target, reference, reference_LD = NULL, reference_U = NULL,
                        flip_cols = FALSE) {
        qc_data <- allele_qc(
            target_data  = target,
            ref_variants = reference,
            col_to_flip  = if (flip_cols) c("Beta", "z") else NULL
        )

        # variant_id below is the *reference panel*'s id (post-QC).
        sumstat_flipped <- qc_data$target_data_qced %>% arrange(pos)
        kept_vars       <- sumstat_flipped$variant_id

        out <- list(sumstat = sumstat_flipped)
        if (!is.null(reference_LD)) out$LD <- reference_LD[kept_vars, kept_vars]
        if (!is.null(reference_U))  out$U  <- reference_U[kept_vars, , drop = FALSE]
        out
    }

    # filter_vars(): restrict an aligned panel to a given shared variant set,
    # keeping LD / U in sync with the sumstat rows.
    filter_vars <- function(input, variants) {
        sumstat   <- input$sumstat %>% filter(variants_id_original %in% variants)
        kept_vars <- sumstat %>% pull(variant_id)

        out <- list(sumstat = sumstat)
        if (!is.null(input$LD)) out$LD <- input$LD[kept_vars, kept_vars, drop = FALSE]
        if (!is.null(input$U))  out$U  <- input$U[kept_vars, , drop = FALSE]
        out
    }

    # In-sample LD matrices (only kg uses LD downstream; UKB_LD is computed for
    # diagnostics / parity but UKB is carried as a sketch panel).
    kg_LD  <- get_cormat(kg_data$bed)
    UKB_LD <- get_cormat(UKB_data$bed)

    # First-pass alignment against the (unsimulated) UKB target.
    kg_input   <- uniform(target = target_tb, reference = kg_data$bim,
                          reference_LD = kg_LD)
    UKB_input  <- uniform(target = target_tb, reference = UKB_data$bim,
                          reference_U  = U_UKB)
    ADSP_input <- uniform(target = target_tb, reference = ADSP_data$bim,
                          reference_U  = U_ADSP)

    # Intersect variants surviving QC across all three panels.
    shared_vars <- Reduce(intersect, list(
        UKB_input$sumstat$variants_id_original,
        kg_input$sumstat$variants_id_original,
        ADSP_input$sumstat$variants_id_original
    ))

    UKB_input  <- filter_vars(UKB_input,  shared_vars)
    kg_input   <- filter_vars(kg_input,   shared_vars)
    ADSP_input <- filter_vars(ADSP_input, shared_vars)

    # =========================================================================
    # 4. Draw a 50k-individual GWAS sample from UKB and simulate the phenotype
    # =========================================================================
    set.seed(123)
    individual_index <- sample(1:nrow(UKB_data$bed), 50000, replace = FALSE)
    Xmat <- UKB_data$bed[individual_index, shared_vars, drop = FALSE]

    # Guard: regions with too few SNPs are not informative for fine-mapping.
    if (ncol(Xmat) < 500) {
        stop("Too few SNPs — terminating job")
    }

    n_samples <- nrow(Xmat)
    p         <- ncol(Xmat)

    ntrait  <- 1
    ncausal <- ${n_causal}
    h2g     <- ${h2g}

    # Sample causal SNPs and effect sizes; build the per-SNP effect vector beta.
    causal_index <- sample_causal_index(Xmat = Xmat, ncausal = ncausal)
    B_mat <- matrix(0, nrow = p, ncol = ntrait)
    beta  <- sim_beta_fix_variant(
        G            = Xmat,
        causal_index = causal_index,
        is_h2g_total = FALSE
    )
    B_mat[, 1] <- beta

    # Simulate the trait: y = X beta + e, with var(e) calibrated to hit h2g.
    phenotype <- sim_multi_traits(
        G            = Xmat,
        B            = B_mat,
        h2g          = h2g,
        is_h2g_total = FALSE
    )$P

    # =========================================================================
    # 5. Compute summary stats and re-align each external panel to them
    # =========================================================================

    # Marginal regression z-scores on the simulated phenotype, with the SNP
    # column re-parsed into (chrom, pos, A2, A1) for QC keys.
    target_tb <- calculate_sumstat(Xmat, as.numeric(phenotype[, 1])) %>%
        separate(
            SNP,
            into   = c("chrom", "pos", "A2", "A1"),
            sep    = ":",
            remove = FALSE
        ) %>%
        mutate(
            chrom      = as.numeric(str_remove(chrom, "^chr")),
            pos        = as.integer(pos),
            original_z = z   # keep an un-flipped copy for diagnostics
        ) %>%
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    # UKB is the in-sample panel, so its sumstat is just the simulated one.
    UKB_input$sumstat <- target_tb

    # Re-QC kg and ADSP against the simulated sumstats with flip_cols = TRUE so
    # z-scores get sign-flipped wherever ref/alt is reversed relative to target.
    kg_input <- uniform(
        target       = target_tb,
        reference    = kg_input$sumstat %>% dplyr::select(-variant_id),
        reference_LD = kg_input$LD,
        flip_cols    = TRUE
    )

    ADSP_input <- uniform(
        target       = target_tb,
        reference    = ADSP_input$sumstat %>% dplyr::select(-variant_id),
        reference_U  = ADSP_input$U,
        flip_cols    = TRUE
    )

    # =========================================================================
    # 6. Save bundled output for downstream SuSiE-RSS / mixture fitting
    # =========================================================================
    output_list <- list(
        kg_input   = kg_input,    # external LD reference (1000G EUR)
        ADSP_input = ADSP_input,  # external sketch reference
        UKB_input  = UKB_input,   # in-sample sketch (target)
        true_vars  = causal_index,
        n_samples  = n_samples,
        n_snp      = p,
        input_path = UKB_path
    )

    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output_list, data_out_path)

In [ ]:
[hetero]
parameter: genofile = paths
# pheno_file: give genotype file (in plink), we can read the genotype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "80G"
parameter: numThreads = 2
parameter: n_causal = 1
# specify the number of causal variants
parameter: h2g = 0.05
parameter: total_h2g = True
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_multi_ref.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    


    sim_beta_fix_variant <- function(G, causal_index, ntrait = 1,
                                 is_h2g_total = TRUE,
                                 h2g_ratio = NULL,
                                 random_sign = FALSE) {
      n_snps <- ncol(G)
      B      <- matrix(0, nrow = n_snps, ncol = ntrait)
    
      if (is_h2g_total) {
        beta <- numeric(n_snps)
        beta[causal_index] <- 1
        for (i in 1:ntrait) B[, i] <- beta
        return(B)
      }
    
      k <- length(causal_index)
    
      # default = old behavior (equal variance contribution per variant)
      if (is.null(h2g_ratio)) {
        h2g_ratio <- rep(1, k)
      } else {
        stopifnot(length(h2g_ratio) == k, all(h2g_ratio > 0))
        # normalize so the largest weight is 1 → that variant becomes the "anchor"
        h2g_ratio <- h2g_ratio / max(h2g_ratio)
      }
    
      var_vector <- apply(as.matrix(G[, causal_index]), 2, var)
    
      # anchor variant = the one with weight 1 (the largest)
      anchor <- which.max(h2g_ratio)[1]
      beta_anchor <- 1
    
      beta <- numeric(n_snps)
      beta_causal <- sqrt(h2g_ratio * beta_anchor^2 * var_vector[anchor] / var_vector)
    
      if (random_sign) {
        beta_causal <- beta_causal * sample(c(-1, 1), k, replace = TRUE)
      }
    
      beta[causal_index] <- beta_causal
    
      for (i in 1:ntrait) B[, i] <- beta
      B
    }

    # -----------------------------------------------------------------------------
    # Parse input: gtex .bim file -> sample index `i`
    # -----------------------------------------------------------------------------
     file = ${_input:r}
    i <- file %>%
        str_remove("\\.bim$") %>%
        str_remove(".*/gtex_")

    # -----------------------------------------------------------------------------
    # Reference panel genotypes
    # -----------------------------------------------------------------------------
    gtex_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/gtex_", i)
    gtex_data <- plink_process(gtex_path)

    UKB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKB_data <- plink_process(UKB_path)

    kg_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/EUR_KG_", i)
    kg_data <- plink_process(kg_path, imiss = 0.1, maf = 0)

    # -----------------------------------------------------------------------------
    # UKB sketch: U_UKB = X_UKB' W   (p_UKB x B, raw allele coding)
    # -----------------------------------------------------------------------------
    W <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/random_vector/result/random_matrix_10000.rds")
    U_UKB <- crossprod(UKB_data$bed, W)

    # -----------------------------------------------------------------------------
    # ADSP sketch (pgen): read X and afreq, recover/scale to build U_ADSP
    # -----------------------------------------------------------------------------
    read_pvar_text <- function(pvar_path) {
        df <- as.data.frame(vroom(pvar_path, comment = "##", show_col_types = FALSE))
        colnames(df)[1] <- "chrom"
        data.frame(
            chrom = as.character(df$chrom),
            id    = as.character(df$ID),
            pos   = as.integer(df$POS),
            A2    = as.character(df$REF),
            A1    = as.character(df$ALT),
            stringsAsFactors = FALSE
        )
    }

    prefix <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes//ADSP_sketch_trimed/result/ADSP_", i)
    pvar   <- paste0(prefix, ".pvar")
    pgen   <- paste0(prefix, ".pgen")
    psam   <- paste0(prefix, ".psam")
    paths  <- list(pvar = pvar, pgen = pgen, psam = psam)

    all_variant_info <- read_tsv(paths$pvar)
    variant_idx      <- seq_len(nrow(all_variant_info))
    variant_info     <- all_variant_info[variant_idx, , drop = FALSE]

    psam <- as.data.frame(read_tsv(paths$psam, show_col_types = FALSE))
    colnames(psam)[1:2] <- c("FID", "IID")

    pgen_obj <- pgenlibr::NewPgen(paths$pgen)
    X <- pgenlibr::ReadList(pgen_obj, variant_subset = variant_idx, meanimpute = FALSE)
    rownames(X) <- psam$IID
    colnames(X) <- variant_info$ID

    chrom_adsp <- all_variant_info$`#CHROM`[1]

    afreq_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/chr", chrom_adsp,
                         "/ADSP.R4.EUR.chr", chrom_adsp, ".afreq")
    afreq <- read_tsv(afreq_path)

    afreq_ordered <- afreq %>% slice(match(colnames(X), ID))
    umin <- afreq_ordered %>% pull(U_MIN)
    umax <- afreq_ordered %>% pull(U_MAX)

    colnames(variant_info) <- c("chrom", "pos", "original_id", "A2", "A1")

    recover_U <- function(U_scaled, Umin, Umax) {
        sweep(
            sweep(U_scaled, 2, 2, "/"),
            2, (Umax - Umin), "*"
        ) + matrix(Umin, nrow(U_scaled), ncol(U_scaled), byrow = TRUE)
    }

    U_ADSP_raw <- X
    U_ADSP_rec <- recover_U(U_ADSP_raw, umin, umax)
    U_ADSP     <- t(scale(U_ADSP_rec, center = TRUE, scale = TRUE))

    ADSP_data <- list(bim = variant_info, U = U_ADSP)

    ## Load finished

    # -----------------------------------------------------------------------------
    # Align all panels to GTEx (target)
    # -----------------------------------------------------------------------------
    target_tb <- gtex_data$bim %>%
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    uniform <- function(target, reference, reference_LD = NULL, reference_U = NULL, flip_cols = FALSE) {
        qc_data <- allele_qc(
            target_data  = target,
            ref_variants = reference,
            col_to_flip  = if (flip_cols) c("Beta", "z") else NULL
        )

        sumstat_flipped <- qc_data$target_data_qced %>% arrange(pos)
        kept_vars       <- sumstat_flipped$variant_id

        out <- list(sumstat = sumstat_flipped)

        if (!is.null(reference_LD)) {
            out$LD <- reference_LD[kept_vars, kept_vars]
        }
        if (!is.null(reference_U)) {
            out$U <- reference_U[kept_vars, , drop = FALSE]
        }

        out
    }

    filter_vars <- function(input, variants) {
        sumstat   <- input$sumstat %>% filter(variants_id_original %in% variants)
        kept_vars <- sumstat %>% pull(variant_id)

        out <- list(sumstat = sumstat)

        if (!is.null(input$LD)) {
            out$LD <- input$LD[kept_vars, kept_vars, drop = FALSE]
        }
        if (!is.null(input$U)) {
            out$U <- input$U[kept_vars, , drop = FALSE]
        }

        out
    }

    gtex_LD <- get_cormat(gtex_data$bed)
    kg_LD   <- get_cormat(kg_data$bed)

    insample_input <- list(
        sumstat = target_tb %>% arrange(pos),
        LD      = gtex_LD[target_tb$variant_id, target_tb$variant_id]
    )
    kg_input   <- uniform(target = target_tb, reference = kg_data$bim,   reference_LD = kg_LD)
    UKB_input  <- uniform(target = target_tb, reference = UKB_data$bim,  reference_U  = U_UKB)
    ADSP_input <- uniform(target = target_tb, reference = ADSP_data$bim, reference_U  = U_ADSP)

    shared_vars <- Reduce(intersect, list(
        insample_input$sumstat$variants_id_original,
        UKB_input$sumstat$variants_id_original,
        kg_input$sumstat$variants_id_original,
        ADSP_input$sumstat$variants_id_original
    ))

    insample_input <- filter_vars(insample_input, shared_vars)
    UKB_input      <- filter_vars(UKB_input,      shared_vars)
    kg_input       <- filter_vars(kg_input,       shared_vars)
    ADSP_input     <- filter_vars(ADSP_input,     shared_vars)

    # -----------------------------------------------------------------------------
    # Simulate phenotype from GTEx genotypes (in-sample)
    # -----------------------------------------------------------------------------
    Xmat <- gtex_data$bed[, shared_vars, drop = FALSE]

    if (ncol(Xmat) < 500) {
        stop("Too few SNPs — terminating job")
    }

    n_samples <- nrow(Xmat)
    p         <- ncol(Xmat)

    ntrait  <- 1
    ncausal <- ${n_causal}
    h2g     <- ${h2g}

    causal_index <- sample_causal_index(Xmat = Xmat, ncausal = ncausal)
    B_mat        <- matrix(0, nrow = p, ncol = ntrait)
    beta <- sim_beta_fix_variant(
      G            = Xmat,
      causal_index = causal_index,
      is_h2g_total = FALSE,
      h2g_ratio    = c(1, 0.6, 0.3)[1:ncausal]   # length must match ncausal
    )

    B_mat[, 1] <- beta

    phenotype <- sim_multi_traits(
        G            = Xmat,
        B            = B_mat,
        h2g          = h2g,
        is_h2g_total = FALSE
    )$P

    ## -----------------------------------------------------------------------------
    ## Simulate summary stats from GTEx samples
    ## -----------------------------------------------------------------------------

    target_tb <- calculate_sumstat(Xmat, as.numeric(phenotype[, 1])) %>%
        separate(
            SNP,
            into = c("chrom", "pos", "A2", "A1"),
            sep  = ":",
            remove = FALSE
        ) %>%
        mutate(
            chrom      = as.numeric(str_remove(chrom, "^chr")),
            pos        = as.integer(pos),
            original_z = z
        ) %>%
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    insample_input$sumstat <- target_tb

    kg_input <- uniform(
        target       = target_tb,
        reference    = kg_input$sumstat %>% dplyr::select(-variant_id),
        reference_LD = kg_input$LD,
        flip_cols    = TRUE
    )

    UKB_input <- uniform(
        target       = target_tb,
        reference    = UKB_input$sumstat %>% dplyr::select(-variant_id),
        reference_U  = UKB_input$U,
        flip_cols    = TRUE
    )

    ADSP_input <- uniform(
        target       = target_tb,
        reference    = ADSP_input$sumstat %>% dplyr::select(-variant_id),
        reference_U  = ADSP_input$U,
        flip_cols    = TRUE
    )
            
    h2g_ratio_vec <- c(1, 0.6, 0.3)[1:ncausal]

    output_list <- list(
        insample_input = insample_input,
        kg_input       = kg_input,
        ADSP_input     = ADSP_input,
        UKB_input      = UKB_input,
        true_vars      = causal_index,
        n_samples      = n_samples,
        n_snp          = p,
        h2g_ratio      = h2g_ratio_vec,
        input_path     = gtex_path
    )

    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output_list, data_out_path)

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_simulation"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run /sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/07.Fig3A_S5A_S5B.Simulate_GTEx_eQTL_GWAS.ipynb multi_ref_simulation \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/*.bim) \
    --n_causal CAUSAL --mem 60G --h2g H2G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/ -j 5
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.02 0.03 0.05 0.1 0.15; do
  # for h2g in 0.02 ; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

       sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="GWAS_sim"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH

sos run /sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/07.Fig3A_S5A_S5B.Simulate_GTEx_eQTL_GWAS.ipynb multi_ref_GWAS \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/*.bim) \
    --n_causal CAUSAL --mem 60G --h2g H2G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/ -j 5
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
    for h2g in 0.005 0.01; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

       sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Hetero_sim"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH

sos run /sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/07.Fig3A_S5A_S5B.Simulate_GTEx_eQTL_GWAS.ipynb hetero \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/*.bim) \
    --n_causal CAUSAL --mem 80G --h2g H2G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/ -j 5
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 2 3; do
  for h2g in 0.075 0.05 0.10 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

       sb "${output_script}"   # use sb if you aliased
  done
done

In [6]:
res = readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/coloc/result/coloc_res_chr1:100794065-101983457.rds")

In [7]:
res

gene,tissue,method,n_insample_cs,n_outsample_cs,replicated_insample,replicated_outsample,B_corrected,max_corr,lambda_pop,Q_art,DCS_max,artifact_flag,sensitivity_flag,reliability_flag,prior_flag,max_abs_z,error
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<lgl>,<lgl>,<dbl>,<chr>
chr1:100794065-101983457,WBC_count,insample_res,1,1,1,1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
chr1:100794065-101983457,WBC_count,insample_eb_mix_res,1,1,1,1,227.95055,NA,0.004385916,0.020183496,1.416621e-04,FALSE,FALSE,FALSE,FALSE,7.178881,NA
chr1:100794065-101983457,WBC_count,susie_rss_res,1,1,1,1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
chr1:100794065-101983457,WBC_count,susie_eb_mix_res,1,1,1,1,199.84705,NA,0.004903827,0.031538977,7.988557e-03,FALSE,FALSE,FALSE,FALSE,7.178881,NA
chr1:100794065-101983457,WBC_count,susie_eb_mix_res_new,1,1,1,1,219.34336,NA,0.004459062,0.031312551,2.558297e-02,FALSE,FALSE,FALSE,NA,7.178881,NA
chr1:100794065-101983457,WBC_count,susie_ser_res,1,1,1,1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
chr1:100794065-101983457,Plateletcrit,insample_res,1,1,1,1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
chr1:100794065-101983457,Plateletcrit,insample_eb_mix_res,1,1,1,1,242.22433,NA,0.004127404,0.018705742,5.966520e-05,FALSE,FALSE,FALSE,FALSE,4.999149,NA
chr1:100794065-101983457,Plateletcrit,susie_rss_res,1,1,1,1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
